<a href="https://colab.research.google.com/github/Drewbits/petrophysical-data-quality-workflow/blob/main/02_Read_and_Inspect_DLIS_File.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Petrophysical Data Quality Workflow

## Notebook 2 – Read and Inspect a DLIS File

**Author:** Andrew Hind

### Business Objective

Digital Log Interchange Standard (DLIS) files can contain multiple logical files, frames, channels, acquisition parameters, and tool metadata. Before extracting curves into a relational database, the internal structure must be inspected and validated.

### Technical Objectives

- Identify available DLIS files
- Select one file for initial testing
- Load the file using `dlisio`
- Inspect logical files, frames, and channels
- Identify depth-indexed curve data
- Convert recorded measurements into a pandas DataFrame
- Validate depth structure and curve completeness
- Identify and handle source null values

### Deliverables

- Selected DLIS test file
- DLIS structural summary
- Frame and channel inventory
- Extracted log DataFrame
- Initial data-ingestion validation

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q dlisio

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

from dlisio import dlis

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("dlisio imported successfully.")

dlisio imported successfully.


In [4]:
extract_folder = Path("/content/drive/MyDrive/Datasets/15_9-19 A")

dlis_files = sorted(
    path for path in extract_folder.rglob("*")
    if path.is_file() and path.suffix.upper() == ".DLIS"
)

print(f"DLIS files found: {len(dlis_files)}")

for path in dlis_files[:10]:
    print(path.relative_to(extract_folder))

DLIS files found: 34
04.COMPOSITE/15_9-F-1/WLC_COMPOSITE_1.DLIS
04.COMPOSITE/15_9-F-1/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
04.COMPOSITE/15_9-F-1 A/WLC_COMPOSITE_1.DLIS
04.COMPOSITE/15_9-F-1 A/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
04.COMPOSITE/15_9-F-1 B/WLC_COMPOSITE_1.DLIS
04.COMPOSITE/15_9-F-1 B/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
04.COMPOSITE/15_9-F-1 C/WLC_COMPOSITE_1.DLIS
04.COMPOSITE/15_9-F-1 C/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
04.COMPOSITE/15_9-F-10/WLC_COMPOSITE_1.DLIS
04.COMPOSITE/15_9-F-10/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS


In [5]:
if not dlis_files:
    raise FileNotFoundError("No DLIS files were found in the extracted dataset.")

test_dlis_path = dlis_files[0]

print("Selected test file:")
print(test_dlis_path.relative_to(extract_folder))
print(f"Size: {test_dlis_path.stat().st_size / 1_000_000:.2f} MB")

Selected test file:
04.COMPOSITE/15_9-F-1/WLC_COMPOSITE_1.DLIS
Size: 2.03 MB


In [6]:
try:
    physical_file = dlis.load(test_dlis_path)
    print("DLIS file loaded successfully.")
    print(f"Logical files found: {len(physical_file)}")
except Exception as error:
    print("The DLIS file could not be loaded.")
    print(f"Error type: {type(error).__name__}")
    print(f"Error message: {error}")

DLIS file loaded successfully.
Logical files found: 1


## 1. Inspect DLIS Structure

A DLIS physical file may contain one or more logical files. Each logical file may contain frames, channels, tools, parameters, and other acquisition metadata. This section inspects the structure before extracting recorded measurements.

In [7]:
logical_file = physical_file[0]

print("Logical file loaded.")
print(f"Frames: {len(logical_file.frames)}")
print(f"Channels: {len(logical_file.channels)}")
print(f"Tools: {len(logical_file.tools)}")
print(f"Parameters: {len(logical_file.parameters)}")

Logical file loaded.
Frames: 1
Channels: 12
Tools: 0
Parameters: 15


## 2. Build the Channel Metadata Catalog

Before extracting numerical measurements, the available channels, descriptions, and units are organized into a structured table.

In [8]:
for i, frame in enumerate(logical_file.frames):
    print(f"Frame {i}")
    print(f"Name: {frame.name}")
    print(f"Description: {frame.description}")
    print(f"Index type: {frame.index_type}")
    print(f"Channels: {len(frame.channels)}")
    print("-" * 50)

Frame 0
Name: 0
Description: None
Index type: BOREHOLE-DEPTH
Channels: 12
--------------------------------------------------


In [9]:
frame = logical_file.frames[0]

for i, channel in enumerate(frame.channels):
    print(f"Channel {i}")
    print(f"Name: {channel.name}")
    print(f"Long name: {channel.long_name}")
    print(f"Units: {channel.units}")
    print("-" * 50)

Channel 0
Name: DEPTH
Long name: 
Units: mm
--------------------------------------------------
Channel 1
Name: GR
Long name: Gamma Ray
Units: gAPI
--------------------------------------------------
Channel 2
Name: CALI
Long name: Caliper
Units: in
--------------------------------------------------
Channel 3
Name: RDEP
Long name: Deep Resistivity
Units: ohm.m
--------------------------------------------------
Channel 4
Name: RMED
Long name: Medium Resistivity
Units: ohm.m
--------------------------------------------------
Channel 5
Name: DEN
Long name: Density
Units: g/cm3
--------------------------------------------------
Channel 6
Name: DENC
Long name: Density Correction
Units: g/cm3
--------------------------------------------------
Channel 7
Name: PEF
Long name: Photoelectric Factor
Units: b/e
--------------------------------------------------
Channel 8
Name: NEU
Long name: Neutron
Units: v/v
--------------------------------------------------
Channel 9
Name: AC
Long name: Sonic Comp

In [10]:
channel_records = []

for channel in frame.channels:
    channel_records.append(
        {
            "mnemonic": channel.name,
            "long_name": channel.long_name,
            "units": channel.units,
        }
    )

channel_df = pd.DataFrame(channel_records)

channel_df

,mnemonic,long_name,units
0,DEPTH,,mm
1,GR,Gamma Ray,gAPI
2,CALI,Caliper,in
3,RDEP,Deep Resistivity,ohm.m
4,RMED,Medium Resistivity,ohm.m
5,DEN,Density,g/cm3
6,DENC,Density Correction,g/cm3
7,PEF,Photoelectric Factor,b/e
8,NEU,Neutron,v/v
9,AC,Sonic Compressional,us/ft


## 3. Extract Recorded Measurements

The selected frame is converted from the DLIS structured array into a pandas DataFrame for subsequent validation and analysis.

In [11]:
frame_data = frame.curves()

print(f"Number of recorded samples: {len(frame_data):,}")
print(f"Fields: {frame_data.dtype.names}")

Number of recorded samples: 34,361
Fields: ('FRAMENO', 'DEPTH', 'GR', 'CALI', 'RDEP', 'RMED', 'DEN', 'DENC', 'PEF', 'NEU', 'AC', 'ROP', 'BS')


In [12]:
log_df = pd.DataFrame(frame_data)

print(f"DataFrame shape: {log_df.shape}")
display(log_df.head())

DataFrame shape: (34361, 13)


,FRAMENO,DEPTH,GR,CALI,RDEP,RMED,DEN,DENC,PEF,NEU,AC,ROP,BS
0,1,197000,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25
1,2,197100,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25
2,3,197200,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25
3,4,197300,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25
4,5,197400,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25


In [13]:
print(f"Rows: {log_df.shape[0]:,}")
print(f"Columns: {log_df.shape[1]}")
print("\nData types:")
print(log_df.dtypes)

Rows: 34,361
Columns: 13

Data types:
FRAMENO      int32
DEPTH        int32
GR         float32
CALI       float32
RDEP       float32
RMED       float32
DEN        float32
DENC       float32
PEF        float32
NEU        float32
AC         float32
ROP        float32
BS         float32
dtype: object


## 4. Validate the Depth Index

The borehole-depth index is checked for sampling consistency, duplicate depths, reversed intervals, and zero-depth increments before evaluating the petrophysical measurements.

In [14]:
depth_diff = log_df["DEPTH"].diff()

print(f"Minimum depth: {log_df['DEPTH'].min()} mm")
print(f"Maximum depth: {log_df['DEPTH'].max()} mm")
print(f"Number of samples: {len(log_df):,}")

print("\nMost common depth increments:")
print(depth_diff.value_counts().head(10))

duplicate_depths = log_df["DEPTH"].duplicated().sum()
reversed_depths = (depth_diff < 0).sum()
zero_increments = (depth_diff == 0).sum()

print(f"\nDuplicate depths: {duplicate_depths}")
print(f"Reversed depth steps: {reversed_depths}")
print(f"Zero depth increments: {zero_increments}")

Minimum depth: 197000 mm
Maximum depth: 3633000 mm
Number of samples: 34,361

Most common depth increments:
DEPTH
100.0    34360
Name: count, dtype: int64

Duplicate depths: 0
Reversed depth steps: 0
Zero depth increments: 0


## 5. Handle Source Null Values

Initial descriptive statistics revealed that the DLIS uses `-999.25` as a numeric null sentinel. Because pandas does not automatically interpret this value as missing data, the sentinel is converted to `NaN` before calculating completeness or descriptive statistics.

In [15]:
NULL_VALUE = -999.25

log_df_clean = log_df.replace(NULL_VALUE, np.nan)

missing_summary = log_df_clean.isna().sum().to_frame("missing_count")

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(log_df_clean) * 100
).round(2)

missing_summary

,missing_count,missing_percent
FRAMENO,0,0.00
DEPTH,0,0.00
GR,188,0.55
CALI,24225,70.50
RDEP,647,1.88
RMED,647,1.88
DEN,24314,70.76
DENC,24218,70.48
PEF,24314,70.76
NEU,24342,70.84


In [16]:
log_df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
FRAMENO,34361.0,1.718100e+04,9919.310636,1.000000,8.591000e+03,1.718100e+04,2.577100e+04,3.436100e+04
DEPTH,34361.0,1.915000e+06,991931.063633,197000.000000,1.056000e+06,1.915000e+06,2.774000e+06,3.633000e+06
GR,34173.0,5.871751e+01,43.643150,0.148800,3.089010e+01,5.818890e+01,8.005780e+01,8.737680e+02
CALI,10136.0,8.588493e+00,0.060485,7.937500,8.578100e+00,8.578100e+00,8.625000e+00,9.742200e+00
RDEP,33714.0,1.502017e+00,1.411322,0.065000,7.215250e-01,1.137800e+00,1.654700e+00,1.534000e+01
RMED,33714.0,1.586192e+00,2.099145,0.093200,7.591000e-01,1.164300e+00,1.730450e+00,1.979690e+02
DEN,10047.0,2.480692e+00,0.135568,1.801100,2.421600e+00,2.533100e+00,2.576900e+00,2.748600e+00
DENC,10143.0,5.150227e-02,0.013705,-0.103500,4.660000e-02,5.170000e-02,5.730000e-02,1.275000e-01
PEF,10047.0,6.668580e+00,1.004383,4.481800,5.956400e+00,6.731200e+00,7.566750e+00,1.071300e+01
NEU,10019.0,1.694280e-01,0.093710,0.033200,1.089000e-01,1.465000e-01,2.086000e-01,6.454000e-01


## 6. Review Cleaned Descriptive Statistics

Descriptive statistics are recalculated after replacing the DLIS null sentinel so that summary metrics reflect valid recorded measurements rather than missing-value placeholders.

## Conclusion

The selected DLIS composite file was successfully loaded and converted into a structured pandas DataFrame.

### Key Findings

- The source contains one logical file and one borehole-depth-indexed frame.
- The frame contains 12 DLIS channels.
- Extraction produced 34,361 recorded samples.
- Depth extends from 197.0 m to 3,633.0 m.
- The data are sampled consistently at 0.1 m intervals.
- No duplicate, reversed, or zero-increment depth values were identified.
- The source uses `-999.25` as a numeric null sentinel.
- Gamma ray, resistivity, ROP, and bit-size measurements provide near-complete coverage of the composite interval.
- Density, neutron, sonic, caliper, PEF, and density-correction measurements cover substantially shorter portions of the composite interval.

### Key Data-Management Lesson

Successful file ingestion does not guarantee analysis-ready data. Structural validation, depth-index verification, source-null handling, and curve-completeness assessment are necessary before petrophysical quality control or interpretation.

### Next Step

The next milestone will build a structured DLIS metadata and curve catalog that can be compared across multiple wells and ultimately loaded into a relational database.